## Notebook 04: Hydrogen Case Study LCA

This notebook runs the environmental and social life cycle assessment calculations for all hydrogen production technologies across all available countries.

**What it does:**
1. Builds functional units for each hydrogen production technology and country combination.
2. Selects all social (PSILCA-derived) and environmental (ReCiPe, EF) impact methods.
3. Runs LCA calculations for each functional unit and extracts node-level contribution scores.
4. Exports results as one CSV per (technology, country) combination.

**Inputs:**
- Brightway project with social flows from notebooks 01 and 03.
- Regionalized hydrogen foreground databases.

**Outputs:**
- Per-activity CSV files in `Data/LCA_contribution_data/Nodes/`, one file per (technology, country) pair, containing scores for all impact methods.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import os
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import bw2data as bd
import bw2calc as bc
from bw2calc import LCA
from bw2analyzer import ContributionAnalysis

## Configuration

In [ ]:
PROJECT = "sense_project"

# Where to read the set of locations from
LOCATION_SOURCE_DB = "01_hydrogen_inventories_regionalized"

# Databases to search for FU activities (priority order)
SEARCH_DBS = [
    "01_hydrogen_inventories_regionalized",
    "01_hydrogen_inventories_regionalized_subset",
    # "Regioinvent",
]

# Activities to include as FUs (exact match on name, ref product, location)
# Each spec = (name, reference_product, amount)
ACTIVITY_SPECS = [
    ("hydrogen production, biomass gasification", "hydrogen production, biomass gasification", 1),
    ("hydrogen production, steam methane reforming of biomethane", "hydrogen production, biomethane", 1),
    ("hydrogen production, PEM electrolysis, green", "hydrogen production, PEM electrolysis, green", 1),
    ("hydrogen production, coal gasification", "hydrogen, gaseous, low pressure", 1),
    ("hydrogen production, steam methane reforming", "hydrogen, gaseous, low pressure", 1),
]

# Top-level method groups to include
TARGET_METHOD_GROUPS = [
    "ReCiPe 2016 v1.03, midpoint (H) regionalized",
    "ReCiPe 2016 v1.03, endpoint (H) regionalized",
    "EF v3.1 regionalized",
    "Local Community", "Society", "Value Chain Actors", "Workers", "Children", "Consumer",
]

# Calculation setup name
SETUP_NAME = "my_batch_run"

# Contribution extraction behavior
LIMIT_TYPE = "number"
LIMIT = 1000

# Output directory for nodes CSVs
OUTPUT_DIR_NODES = Path(r"..\Data\LCA_contribution_data\Nodes")
OUTPUT_DIR_NODES.mkdir(parents=True, exist_ok=True)

## Helper Functions

In [ ]:
def normalize_filename(name: str) -> str:
    """Normalize activity names for safe filenames."""
    return re.sub(r"[^\w\-_\. ]", "_", name.lower().strip()).replace(" ", "_")


def build_setup_exact(
    project_name: str,
    location_source_db: str,
    search_dbs: Sequence[str],
    activity_specs: Sequence[Tuple[str, str, float]],
    target_groups: Sequence[str],
    setup_name: str,
    strict_unique: bool = True,
    verbose: bool = True,
) -> Tuple[str, List[dict], List[tuple], dict, list]:
    """
    Build calculation_setup with:
      - locations from `location_source_db`
      - exact search over `search_dbs` for (name, reference product, location)
      - no fallbacks
      - methods filtered by top-level group (method[0] in target_groups)

    Returns: (setup_name, func_units, methods, diagnostics, duplicates)
    """
    bd.projects.set_current(project_name)

    # Locations
    src_db = bd.Database(location_source_db)
    locations = sorted({act.get("location") for act in src_db})
    if verbose:
        print(f"🌍 Locations from '{location_source_db}' (n={len(locations)}).")

    # Index DBs once
    def index_db(dbname: str):
        idx = defaultdict(list)
        db = bd.Database(dbname)
        for act in db:
            key = (act.get("name"), act.get("reference product"), act.get("location"))
            idx[key].append(act)
        return idx

    db_indices = {dbn: index_db(dbn) for dbn in search_dbs}
    if verbose:
        print(f"🗂️ Indexed DBs: {', '.join(search_dbs)}")

    def exact_hits(name: str, ref_prod: str, loc: str):
        hits = []
        for dbn in search_dbs:
            hits.extend([(dbn, a) for a in db_indices[dbn].get((name, ref_prod, loc), [])])
        return hits

    func_units: List[dict] = []
    duplicates = []
    diagnostics = {}

    for loc in locations:
        found_count = 0
        not_found_specs = []
        chosen_keys_for_loc = []

        for name, ref_prod, amount in activity_specs:
            hits = exact_hits(name, ref_prod, loc)
            if not hits:
                not_found_specs.append(f"{name} [{ref_prod}]")
                continue

            if strict_unique and len(hits) != 1:
                duplicates.append((loc, name, ref_prod, len(hits), ",".join([h[0] for h in hits])))

            chosen = hits[0][1]  # first by db priority order
            func_units.append({chosen.key: amount})
            found_count += 1
            chosen_keys_for_loc.append(chosen.key)

        diagnostics[loc] = {
            "found": found_count,
            "not_found": len(not_found_specs),
            "missing_specs": not_found_specs,
            "found_keys": chosen_keys_for_loc,
        }

    # Impact methods
    impact_methods = [m for m in bd.methods if m[0] in set(target_groups)]
    if not impact_methods:
        raise ValueError(f"No methods matched the top-level groups: {target_groups}")

    bd.calculation_setups[setup_name] = {"inv": func_units, "ia": impact_methods}

    if verbose:
        print(f"\n📦 Functional units created: {len(func_units)}")
        print(f"🧪 Methods selected: {len(impact_methods)}")
        print(f"✅ Saved calculation setup as '{setup_name}'")

        # compact diagnostics
        for loc, d in list(diagnostics.items())[:20]:
            print(f"  - {loc}: found={d['found']}, not_found={d['not_found']}")
        if len(diagnostics) > 20:
            print(f"  … (+{len(diagnostics)-20} more locations)")

        if duplicates:
            print("\n⚠️ Duplicates (kept first by DB priority order):")
            for loc, n, r, cnt, dbs in duplicates[:10]:
                print(f"  - {loc} | {n} | {r} | {cnt} hits | {dbs}")
            if len(duplicates) > 10:
                print(f"  … and {len(duplicates)-10} more")

    return setup_name, func_units, impact_methods, diagnostics, duplicates


def warmup_lci_across_fu_databases(func_units: Sequence[dict]) -> LCA:
    """
    Initialize an LCA object using a tiny demand across all databases represented in the FUs.
    This improves speed/stability of subsequent redo_lci calls.
    """
    rep_by_db = {}
    for fu in func_units:
        k = next(iter(fu))
        rep_by_db.setdefault(k[0], k)

    warmup_demand = {k: 1e-12 for k in rep_by_db.values()}
    lca = LCA(warmup_demand)
    lca.lci(factorize=True)
    return lca

## Build Calculation Setup

In [ ]:
setup_name, func_units, methods, diagnostics, duplicates = build_setup_exact(
    project_name=PROJECT,
    location_source_db=LOCATION_SOURCE_DB,
    search_dbs=SEARCH_DBS,
    activity_specs=ACTIVITY_SPECS,
    target_groups=TARGET_METHOD_GROUPS,
    setup_name=SETUP_NAME,
    strict_unique=True,
    verbose=True,
)

## Run Batch LCA

In [ ]:
bd.projects.set_current(PROJECT)
method_labels = [" | ".join(m) for m in methods]

print("\n🔄 Initializing LCA with warm-up demand spanning all FU databases...")
lca = warmup_lci_across_fu_databases(func_units)

# Precompute characterization matrices
cf_matrices = {}
for method in methods:
    lca.switch_method(method)
    cf_matrices[method] = lca.characterization_matrix.copy()

ca = ContributionAnalysis()
reverse_proc_dict = lca.reverse_dict()[1]
activity_cache: Dict[tuple, dict] = {}

def get_metadata(key: tuple) -> dict:
    if key not in activity_cache:
        act = bd.get_activity(key)
        activity_cache[key] = {
            "name": act["name"],
            "ref_product": act["reference product"],
            "location": act["location"],
            "unit": act.get("unit", ""),
            "database": act["database"],
        }
    return activity_cache[key]

print("\n🔬 Running LCA, extracting node contributions, writing CSVs...\n")

project_name_for_files = bd.projects.current

for fu in tqdm(func_units, desc="Processing FUs (nodes-only)"):
    lca.redo_lci(fu)

    fu_key = next(iter(fu))
    meta_fu = get_metadata(fu_key)
    filename_base = f"{project_name_for_files}_{normalize_filename(meta_fu['name'])}_{meta_fu['location']}"

    # Base rows for output
    row_score = {"index": "Score", "name": "", "reference product": "", "location": "", "unit": "", "database": ""}
    rest_plus: Dict[str, float] = {}
    rest_minus: Dict[str, float] = {}

    contributor_rows: Dict[str, dict] = {}

    for j, method in enumerate(methods):
        label = method_labels[j]

        # Use precomputed CF matrix
        lca.characterization_matrix = cf_matrices[method]
        lca.lcia_calculation()

        contrib_vector = np.array(lca.characterized_inventory.sum(axis=0)).flatten()
        row_score[label] = lca.score

        # Top contributors for this method
        sorted_contribs = ca.sort_array(
            contrib_vector,
            limit=LIMIT,
            limit_type=LIMIT_TYPE,
            total=np.abs(contrib_vector).sum()
        )
        top_indices = {int(idx) for _, idx in sorted_contribs}

        # Fill contributor table
        for value, idx in sorted_contribs:
            key = reverse_proc_dict[int(idx)]
            meta = get_metadata(key)
            index_label = f"{meta['name']} | {meta['ref_product']} | {meta['location']}"

            if index_label not in contributor_rows:
                contributor_rows[index_label] = {
                    "index": index_label,
                    "name": meta["name"],
                    "reference product": meta["ref_product"],
                    "location": meta["location"],
                    "unit": meta["unit"],
                    "database": meta["database"],
                }
            contributor_rows[index_label][label] = value

        # Rest (+/-): everything not in top_indices
        for idx, val in enumerate(contrib_vector):
            if idx not in top_indices and val != 0:
                if val > 0:
                    rest_plus[label] = rest_plus.get(label, 0.0) + float(val)
                else:
                    rest_minus[label] = rest_minus.get(label, 0.0) + float(val)

    # Build output rows
    rows = [
        row_score,
        {"index": "Rest (+)", "name": "", "reference product": "", "location": "", "unit": "", "database": "",
         **{k: rest_plus.get(k, "") for k in method_labels}},
        {"index": "Rest (-)", "name": "", "reference product": "", "location": "", "unit": "", "database": "",
         **{k: rest_minus.get(k, "") for k in method_labels}},
    ]
    rows.extend(contributor_rows.values())

    out_path = OUTPUT_DIR_NODES / f"{filename_base}.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)

print("\n✅ All FUs processed. Nodes CSVs written.")